In [3]:
import logging

import carla

import carla_extension
from carla_extension.manager import CarlaSimulatorManager


logging.basicConfig(level=logging.INFO)
carla_config = carla_extension.CarlaConfig(low_quality_mode=True, waiting_time=5.0)

manager = CarlaSimulatorManager(carla_config)

client = manager.startup()\
                .change_world("Town05_Opt")\
                .client

INFO:root:carla has already been running on port 2000
INFO:root:World(id=13337151315175716522)


In [4]:
# 生成车辆

# 设置车型
blueprint_library = client.get_world().get_blueprint_library()
vehicle_bp = blueprint_library.find("vehicle.nissan.micra")

# 位置
# 初始点(朝北,270度)
spawn_point = carla.Transform(carla.Location(x=-1, y=-38, z=0.2), carla.Rotation(yaw=270))
# 车位 (朝西, 180度)
end_point = carla.Transform(carla.Location(x=4, y=-30, z=0.1), carla.Rotation(yaw=180))

# 生成车辆
ego: carla.Vehicle = client.get_world().spawn_actor(vehicle_bp, spawn_point)  # type: ignore

In [5]:

# 绑定摄像头
camera_delta = carla.Location(x=0, y=0, z=30)  # 在车辆正上方30米
spectator = client.get_world().get_spectator()

# 让摄像头跟随车辆 (需要写在 for 循环里)
camera_transform = ego.get_transform()
camera_transform.location = ego.get_location() + camera_delta
camera_transform.rotation = carla.Rotation(yaw=ego.get_transform().rotation.yaw, pitch=-90)  # 让摄像头朝下

spectator.set_transform(camera_transform)

In [6]:
# 微调车辆位置
spawn_point.rotation.yaw = 270
ego.set_transform(end_point)
client.get_world().wait_for_tick()
# 让摄像头跟随车辆 (需要写在 for 循环里)
camera_transform = ego.get_transform()
camera_transform.location = ego.get_location() + camera_delta
camera_transform.rotation = carla.Rotation(yaw=camera_transform.rotation.yaw, pitch=-90)  # 让摄像头朝下

spectator.set_transform(camera_transform)

In [ ]:
control = carla.VehicleControl(throttle=0.0,
                               steer=0.0,
                               brake=1.0,
                               hand_brake=False,
                               reverse=False,
                               manual_gear_shift=True,
                               gear=1
                               )
ego.apply_control(control)
client.get_world().wait_for_tick()